In [44]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import numpy as np
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from ucimlrepo import fetch_ucirepo
from features import correlation_groups, choose_columns

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
root = Path.cwd()
data_dir = root.parent._str + "\\model" + "\\data"
metadata_path = root.parent._str + "\\model" + "\\metadata.json"

# 1) Análise de Dados

In [48]:
# Busca do Dataset
rt_iot2022 = fetch_ucirepo(id=942)

# União das features com as targets
df_iot2022 = pd.concat([rt_iot2022.data.features, rt_iot2022.data.targets], axis=1)

# Definição da coluna target
target_column = rt_iot2022.data.targets.columns[0]

print(f'Shape: {df_iot2022.shape}')
print(f'Target: {target_column}\n')
display(df_iot2022.head())

Shape: (123117, 84)
Target: Attack_type



,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,down_up_ratio,fwd_header_size_tot,fwd_header_size_min,fwd_header_size_max,bwd_header_size_tot,bwd_header_size_min,bwd_header_size_max,flow_FIN_flag_count,flow_SYN_flag_count,flow_RST_flag_count,fwd_PSH_flag_count,bwd_PSH_flag_count,flow_ACK_flag_count,fwd_URG_flag_count,bwd_URG_flag_count,flow_CWR_flag_count,flow_ECE_flag_count,fwd_pkts_payload.min,fwd_pkts_payload.max,fwd_pkts_payload.tot,fwd_pkts_payload.avg,fwd_pkts_payload.std,bwd_pkts_payload.min,bwd_pkts_payload.max,bwd_pkts_payload.tot,bwd_pkts_payload.avg,bwd_pkts_payload.std,flow_pkts_payload.min,flow_pkts_payload.max,flow_pkts_payload.tot,flow_pkts_payload.avg,flow_pkts_payload.std,fwd_iat.min,fwd_iat.max,fwd_iat.tot,fwd_iat.avg,fwd_iat.std,bwd_iat.min,bwd_iat.max,bwd_iat.tot,bwd_iat.avg,bwd_iat.std,flow_iat.min,flow_iat.max,flow_iat.tot,flow_iat.avg,flow_iat.std,payload_bytes_per_second,fwd_subflow_pkts,bwd_subflow_pkts,fwd_subflow_bytes,bwd_subflow_bytes,fwd_bulk_bytes,bwd_bulk_bytes,fwd_bulk_packets,bwd_bulk_packets,fwd_bulk_rate,bwd_bulk_rate,active.min,active.max,active.tot,active.avg,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,0.281148,0.156193,0.437341,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0,33,76,8.444444,13.115936,0,23,32,6.4,9.555103,0,33,108,7.714286,11.618477,761.985779,29729182.96,32011597.87,4001449.734,10403073.63,4438.877106,1511693.954,2.026391e+06,506597.757339,680406.147100,761.985779,29729182.96,32011597.87,2.462431e+06,8.199747e+06,3.373777,3.0,1.666667,25.333333,10.666667,0.0,0.0,0.0,0.0,0.0,0.0,2282414.913,2282414.913,2282414.913,2282414.913,0.0,29729182.96,29729182.96,29729182.96,29729182.96,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,0.282277,0.156821,0.439097,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0,33,76,8.444444,13.115936,0,23,32,6.4,9.555103,0,33,108,7.714286,11.618477,247.001648,29855277.06,31883584.02,3985448.003,10463455.56,4214.048386,1576435.804,1.876261e+06,469065.249000,741351.686200,247.001648,29855277.06,31883584.02,2.452583e+06,8.242459e+06,3.387323,3.0,1.666667,25.333333,10.666667,0.0,0.0,0.0,0.0,0.0,0.0,2028306.961,2028306.961,2028306.961,2028306.961,0.0,29855277.06,29855277.06,29855277.06,29855277.06,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,0.280164,0.155647,0.435811,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0,33,74,8.222222,12.852799,0,21,30,6.0,8.689074,0,33,104,7.428571,11.229866,283.956528,29842149.02,32124053.00,4015506.625,10442377.92,2456.903458,1476048.946,2.013770e+06,503442.466300,660344.360027,283.956528,29842149.02,32124053.00,2.471081e+06,8.230593e+06,3.237450,3.0,1.666667,24.666667,10.000000,0.0,0.0,0.0,0.0,0.0,0.0,2281903.982,2281903.982,2281903.982,2281903.982,0.0,29842149.02,29842149.02,29842149.02,29842149.02,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,0.281593,0.156440,0.438033,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0,33,74,8.222222,12.852799,0,21,30,6.0,8.689074,0,33,104,7.428571,11.229866,288.963318,29913774.97,31961063.15,3995132.893,10482528.22,3933.906555,1551892.042,1.883784e+06,470946.013927,724569.317900,288.963318,29913774.97,31961063.15,2.458543e+06,8.257786e+06,3.253959,3.0,1.666667,24.666667,10.000000,0.0,0.0,0.0,0.0,0.0,0.0,2047288.179,2047288.179,2047288.179,2047288.179,0.0,29913774.97,29913774.97,29913774.97,29913774.97,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,0.282111,0.156728,0.438839,0.555556,296,32,40,168,32,40,0,2,1,3,3,13,0,0,0,0,0,33,76,8.444444,13.115936,0,23,32,6.4,9.555103,0,33,108,7.714286,11.618477,387.907028,29814704.90,31902361.87,3987795.234,10447019.10,3005.027771,1632083.178,1.935984e+06,483996.033700,768543.390313,387.907028,2981

# 2) Limpeza de Dados

In [49]:
# Remoção das colunas constantes
constant_columns = [c for c in df_iot2022.columns if df_iot2022[c].nunique() <= 1]
df_iot2022 = df_iot2022.drop(columns=constant_columns, axis=1)

In [50]:
# Remoção dos ids
df_iot2022 = df_iot2022.drop(columns=["id.orig_p", "id.resp_p"], axis=1)

# 3) Split Estratificado

In [51]:
X = df_iot2022.drop(columns=[target_column])
y = df_iot2022[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4) Avaliação de Redundâncias

In [52]:
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=np.number).columns.tolist()

In [53]:
corr_X_train = X_train[numeric_columns].corr(method='spearman').abs()
group_list = correlation_groups(corr_matrix=corr_X_train)

# Mutual Info
Y_train_encoded = LabelEncoder().fit_transform(y_train)
mi_train = mutual_info_classif(X_train[numeric_columns], Y_train_encoded, random_state=43)
importances = dict(zip(numeric_columns, mi_train))


_, drop_columns = choose_columns(group_list=group_list, importances = importances)
print(f'Quantidade de colunas a remover: {len(drop_columns)}')

X_train = X_train.drop(columns=drop_columns, axis=1)
X_test = X_test.drop(columns=drop_columns, axis=1)

numeric_columns = [c for c in numeric_columns if c in X_train.columns]

Quantidade de colunas a remover: 47


# 5) Encoding das Categorias

In [54]:
X_train_encoded = pd.get_dummies(data=X_train, columns=categorical_columns, drop_first=True)
X_test_encoded = pd.get_dummies(data=X_test, columns=categorical_columns, drop_first=True)

# 6) Salvamento de Dados

In [55]:
train_out = X_train_encoded.copy()
test_out = X_test_encoded.copy()

train_out[target_column] = y_train.values 
test_out[target_column] = y_test.values

In [ ]:
metadata = {
    "dataset": "RT-IoT2022",
    "target": target_column,
    "random_state": 42,
    "n_train": len(train_out),
    "n_test": len(test_out),
    "n_features":  X_train_encoded.shape[1],
    "classes": sorted(y.unique().tolist()),
    "feature_names": X_train_encoded.columns.tolist(),
    "id_to_remove": ["id.orig_p", "id.resp_p"],
    "redundant_colunns": sorted(drop_columns),
    "original_categorical": categorical_columns
}

In [57]:
train_out.to_csv(data_dir + "\\train.csv", index=False)
test_out.to_csv(data_dir + "\\test.csv", index=False)

In [ ]:
with open(metadata_path, mode='w') as file:
    json.dump(metadata, file, indent=2, default=str)

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\rpf97\\OneDrive\\Documentos\\Projetos_VSCODE\\Ciencia_Dados_(2026_2027)\\Projetos\\IoT-Guard\\artifacts\\model'